# Pseudonymizer - train on the «base» dataset on a machine with several GPUs (vast.ai)

The vast.ai twin of `colab_train_base.ipynb`: the same models, settings, run names and leaderboard, on `MyDrive/ai-camp/silver/` (tag `v7`) - `data/silver/` of the repo: 16k documents, 83M characters. `vast_train_large.ipynb` does the same on the «large» dataset; the two can run at the same time on one machine - a GPU trains one model at a time.

Once per instance, on the laptop: `train/scripts/vast.sh push` (the code and the test set), `train/scripts/vast.sh drive` (access to Google Drive), `train/scripts/vast.sh hf` (the Hugging Face token, write access). Then `train/scripts/vast.sh ssh`, open https://localhost:8080 and this file in `/workspace/ai-camp/train/`. After a change of the code run `vast.sh push` again.

What differs from Colab:

- the runs of a launch are trained in parallel, one per GPU, the large models first; the training runs in tmux, so the notebook may be closed;
- Google Drive is not mounted: the dataset is the one of `MyDrive/ai-camp/<folder>` when that folder exists there (otherwise the one copied from the laptop by `vast.sh push`), every run is copied to `MyDrive/ai-camp/runs/<run>/` right after training, the leaderboard files are exchanged in cell 6;
- the leaderboard predicts the test set only with the models that are on this machine; the earlier runs of Drive enter it with the test results saved there, so a run of Drive that was scored on another test file or decoder stays in the history tables only. For the same reason `TOP_N` ranks the history as it is.

**Nothing is overwritten**: a launch gets the name `<date>_r<N>` (Kyiv time; the numbers taken on Drive are skipped), a run the folder `<model>_v7_<launch>[_s<seed>]`, the plan is `runs/launches/<launch>_v7.json`. To continue an interrupted launch put its name into `LAUNCH` and run all cells again: the finished runs are skipped.

**Hugging Face.** Unlike in Colab, with `HF_PUBLISH = True` every model trained on the whole dataset is published right after training as the **public** repo `<your HF account>/<run>` (licence mit, a model card with the base model and the sources of the data); a smoke run (`N_DOCS`) is never published. Cell 7 publishes one run by hand, e.g. under another name.

**Progress.** Cell 4 follows the log; cell 4a (or `train/scripts/vast.sh status` on the laptop) shows every run: GPU, share of the steps, epoch, time left, dev F1 after each epoch, whether it reached Drive and Hugging Face, and the error of a failed run.

**Read the table «Averages over seeds».** Two identical runs differ by up to 3.5 points on the test set: a difference smaller than the spread (max − min) inside a group is noise.

In [ ]:
#@title 1. Code and the machine
import os, subprocess, sys
os.chdir('/workspace/ai-camp/train')  # the code is copied here from the laptop: train/scripts/vast.sh push
if os.path.exists('/venv/main/bin') and '/venv/main/bin' not in os.environ['PATH']:
    os.environ['PATH'] = '/venv/main/bin:' + os.environ['PATH']  # the Python environment of the vast.ai image
os.environ.update(TRANSFORMERS_VERBOSITY='error', PYTHONWARNINGS='ignore', TOKENIZERS_PARALLELISM='false', HF_HUB_DISABLE_PROGRESS_BARS='1')
!bash scripts/vast_setup.sh

In [ ]:
#@title 2. Google Drive: the dataset and the history of the runs
VARIANT = 'base'
sys.path.insert(0, 'scripts')
from launch import VARIANTS, file_sha1
TAG, FOLDER = VARIANTS[VARIANT]['tag'], VARIANTS[VARIANT]['folder']
DRIVE_ON = 'gdrive:' in subprocess.run(['rclone', 'listremotes'], capture_output=True, text=True).stdout.split()
if DRIVE_ON:
    # as in Colab, the training data is the one of MyDrive/ai-camp/<folder>
    !bash scripts/drive.sh data $FOLDER
    !bash scripts/drive.sh history
else:
    print('Google Drive is NOT connected: the runs will stay on this machine and the data is the one copied by vast.sh push.\n'
          'To connect it, on the laptop: train/scripts/vast.sh drive')
for need in ('train.parquet', 'dev.parquet', 'dev.jsonl'):
    assert os.path.exists(f'../data/{FOLDER}/{need}'), f'{need} is missing in data/{FOLDER} - upload it to MyDrive/ai-camp/{FOLDER} or run vast.sh push'
assert os.path.exists('../data/evaluate.py') and os.path.exists('../data/test/court_cases.jsonl'), 'data/evaluate.py or the test set is missing: vast.sh push'
print(f"OK: data/{FOLDER}, train.parquet {file_sha1(f'../data/{FOLDER}/train.parquet')}, tag {TAG}")

In [ ]:
#@title 3. Models and the plan of this launch
TOP_N = 0            # besides MODELS, add this many of the best models of the test history (0 = none)
LAUNCH = ""          # empty = a new launch (today's date + next number); "2026-10-09_r1" continues that launch
SEEDS = "42,43"  # comma-separated; every model is trained once per seed and the leaderboard averages them
N_DOCS = 0           # 0 = all; e.g. 2000 for a smoke run
CPU_TIMING_DOCS = 30 # test docs for the CPU speed estimate (0 = skip)
HF_PUBLISH = True    # publish every trained model as the PUBLIC repo <your HF account>/<run> (token: vast.sh hf); runs with N_DOCS are never published
# the models are MODELS of scripts/launch.py: wechsel_base and modern_liberta_large_w2048 (windows of 2048 tokens)
import json, re
args = ['--variant', VARIANT] + (['--launch', LAUNCH] if LAUNCH else ['--seeds', SEEDS, '--n-docs', str(N_DOCS), '--top-n', str(TOP_N)])
out = subprocess.run(['python', 'scripts/launch.py', *args, '--plan-only'], capture_output=True, text=True)
print(out.stdout, out.stderr[-2000:])
LAUNCH = re.search(r'LAUNCH = (\S+)', out.stdout).group(1)
plan = json.load(open(f'runs/launches/{LAUNCH}_{TAG}.json'))
JOBS = plan['jobs']; RUNS = [j['run'] for j in JOBS]
TIMING_RUNS = list({j['name']: j['run'] for j in reversed(JOBS)}.values())  # CPU speed depends on the model, not on the data or seed

In [ ]:
#@title 4. Train: one run per GPU; every run goes to runs/<model>_<tag>_<launch>[_s<seed>] and to Drive right after training
# The training runs in tmux, not in this kernel: a closed tab or a stopped cell (the ■ button) does not stop it. Run the
# cell again to follow the log, or stop it and use the next cell; to stop the training itself: !tmux kill-session -t =train_<tag>
import time
session, log = f'train_{TAG}', f'runs/launch_{TAG}.log'
# '=': the exact session name; without it tmux takes train_v7 for train_v7large (it matches by prefix)
alive = lambda: subprocess.run(['tmux', 'has-session', '-t', '=' + session], capture_output=True).returncode == 0
if alive():
    print(f'a launch of {TAG} is already running - following its log')
else:
    py = subprocess.run('command -v python', shell=True, capture_output=True, text=True).stdout.strip()
    subprocess.run(['tmux', 'new-session', '-d', '-s', session, f"{py} scripts/launch.py --variant {VARIANT} --launch {LAUNCH}{'' if HF_PUBLISH else ' --no-publish'} 2>&1 | tee -a {log}"], check=True)
    time.sleep(3)
with open(log, encoding='utf-8') as f:
    f.seek(max(0, os.path.getsize(log) - 3000))
    while True:
        running = alive()
        print(f.read(), end='', flush=True)
        if not running:
            break
        time.sleep(20)
!python scripts/launch.py --variant $VARIANT --launch $LAUNCH --status

In [ ]:
#@title 4a. Progress - run at any time (stop cell 4 with ■ first: the training goes on)
# per run: waiting / running (GPU, % of the steps, epoch, time left, dev F1 of the finished epochs) / done (dev F1, minutes,
# Drive, Hugging Face) / FAILED (the error). The same from the laptop: train/scripts/vast.sh status
!python scripts/launch.py --variant base --status
!python scripts/launch.py --variant large --status
!nvidia-smi --query-gpu=index,utilization.gpu,memory.used,memory.total --format=csv,noheader

In [ ]:
#@title 5. CPU inference timing (use it for ranking, not absolute numbers)
p = 'runs/cpu_timing_leaderboard.json'
timing = json.load(open(p)) if os.path.exists(p) else {}  # timings of earlier launches are kept
if CPU_TIMING_DOCS:
    !head -n $CPU_TIMING_DOCS ../data/test/court_cases.jsonl > /tmp/test_small.jsonl
    for run in [r for r in TIMING_RUNS if os.path.exists(f'runs/{r}/model')]:  # one run per model
        t0 = time.time()
        subprocess.run(['python','scripts/predict.py','--model',f'runs/{run}/model','--input','/tmp/test_small.jsonl','--output',f'/tmp/{run}_cpu.jsonl','--device','cpu','--threads','4','--batch','4'], capture_output=True)
        timing[run] = (time.time() - t0) / CPU_TIMING_DOCS
        print(f'{run:40s} {timing[run]:5.2f} s/doc -> test (240 docs) ~ {timing[run]*240/60:4.1f} min')
    json.dump(timing, open(p, 'w'))

In [ ]:
#@title 6. Leaderboard on the organizers' test set: this launch + the history of Drive -> runs/leaderboard.md, also on Drive
from IPython.display import Markdown, display
!bash scripts/drive.sh history
!python scripts/leaderboard.py --drive . --name leaderboard --all
!bash scripts/drive.sh results
display(Markdown(open('runs/leaderboard.md', encoding='utf-8').read()))

In [ ]:
#@title 7. Publish ONE model to Hugging Face - by hand: choose the run, set PUBLISH = True, run the cell
PUBLISH = False      # False = only the list of the runs is shown
RUN = ""             # the run to publish: a folder of train/runs or of MyDrive/ai-camp/runs, as the list below and the leaderboard name it
HF_USER = ""         # your HF username or org
HF_REPO = ""         # name of the model on the Hub; empty = the name of the run
HF_PRIVATE = False   # False = public repo, as scripts/publish.py does by default
# licence id of the model card. The task rules ask for an open licence; mit is the licence of the training corpus
# lang-uk/court-decisions-uk and of the wechsel base model. A card that already exists on the Hub is not rewritten.
HF_LICENSE = "mit"
here = sorted(r for r in os.listdir('runs') if os.path.exists(f'runs/{r}/model/config.json'))
there = [r.rstrip('/') for r in subprocess.run(['rclone', 'lsf', 'gdrive:runs', '--dirs-only'], capture_output=True, text=True).stdout.split()] if globals().get('DRIVE_ON') else []
mine = set(globals().get('RUNS', []))  # the runs of this launch, when cell 3 was run
print(f'{len(here)} runs with a model on this machine (* = this launch); + = only on Drive:')
for r in sorted(set(here) | set(there) - {'launches'}):
    print(f"  {'*' if r in mine else ' ' if r in here else '+'} {r}")
RUN, HF_USER, HF_REPO = RUN.strip(), HF_USER.strip(), HF_REPO.strip()
# the token (write access) is stored by train/scripts/vast.sh hf; `hf auth login` on the instance puts it under HF_HOME
token_files = [os.path.expanduser('~/.cache/huggingface/token'), os.path.join(os.environ.get('HF_HOME', '/workspace/.hf_home'), 'token')]
token = os.environ.get('HF_TOKEN') or next((open(f).read().strip() for f in token_files if os.path.exists(f)), '')
if not PUBLISH:
    print('\nPUBLISH is False: nothing is published')
elif RUN not in here and RUN not in there:
    print(f'\nNOT published: no run «{RUN}» - copy a name from the list into RUN')
elif not HF_USER:
    print('\nNOT published: HF_USER is empty')
elif not token:
    print('\nNOT published: no Hugging Face token on this machine - on the laptop: train/scripts/vast.sh hf')
else:
    if RUN not in here:
        !rclone copy gdrive:runs/$RUN runs/$RUN
    tag = re.search(r'_(v\d+[a-z]*)_\d{4}-', RUN)  # wechsel_base_v7large_2026-10-09_r1 -> v7large
    data = f'the synthetic dataset `{tag.group(1)}`' if tag else 'a synthetic dataset'
    note = (f'Trained on {data}: court decisions of the Unified State Register of Court Decisions of Ukraine whose anonymised '
            'fragments are filled with generated values. The generated addresses come from the Ukrposhta directory and from '
            'OpenStreetMap (© OpenStreetMap contributors, ODbL).')
    repo = f'{HF_USER}/{HF_REPO or RUN}'
    print(f"\npublishing {RUN} -> https://huggingface.co/{repo} ({'private' if HF_PRIVATE else 'PUBLIC'}, licence {HF_LICENSE}) ...", flush=True)
    cmd = ['python', 'scripts/publish.py', '--model', f'runs/{RUN}', '--repo', repo, '--license', HF_LICENSE, '--note', note,
           '--message', f'Upload {RUN}'] + (['--private'] if HF_PRIVATE else [])
    r = subprocess.run(cmd, capture_output=True, text=True, env={**os.environ, 'HF_TOKEN': token})
    print(f'published: {r.stdout.strip()}' if r.returncode == 0 else f'NOT published: {(r.stderr or r.stdout).strip()[-400:]}')